# Module 3 • Classical Natural Language Processing

# Lesson 18 • Topic Modeling with Latent Dirichlet Allocation

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Difficulty:** Intermediate  
**Estimated study time:** 100–130 minutes

---

## Scope

This lesson introduces unsupervised topic modeling with Latent Dirichlet
Allocation. It covers document-term matrices, topic-word distributions,
document-topic distributions, preprocessing, topic interpretation, topic
assignment, model comparison, stability, perplexity, coherence limitations,
and multilingual considerations.

## Learning Objectives

After completing this lesson, the learner should be able to:

- explain the goal of topic modeling;
- distinguish topic modeling from classification and clustering;
- describe the assumptions of Latent Dirichlet Allocation;
- prepare a count-based document-term matrix;
- train an LDA model with scikit-learn;
- inspect topic-word distributions;
- inspect document-topic distributions;
- assign dominant topics to documents;
- interpret topics cautiously;
- compare several values for the number of topics;
- explain perplexity and its limitations;
- assess topic stability and reproducibility;
- identify common preprocessing and interpretation errors;
- discuss Arabic and multilingual topic modeling.

## Table of Contents

1. What Is Topic Modeling?
2. Topic Modeling Versus Classification
3. The LDA Intuition
4. Example Corpus
5. Preparing the Document-Term Matrix
6. Why LDA Uses Counts
7. Training an LDA Model
8. Topic-Word Distributions
9. Displaying Top Terms
10. Document-Topic Distributions
11. Dominant Topic Assignment
12. Interpreting and Naming Topics
13. Topic Prevalence
14. Choosing the Number of Topics
15. Perplexity
16. Topic Coherence
17. Topic Stability
18. Hyperparameters
19. Preprocessing Decisions
20. New-Document Inference
21. Error Analysis and Common Failures
22. Multilingual and Arabic Considerations
23. Reproducibility and Reporting
24. Knowledge Check
25. Exercises
26. Summary and Next Lesson

# 1. What Is Topic Modeling?

**Topic modeling** is an unsupervised method for discovering recurring themes
in a collection of documents.

It can support:

- exploratory corpus analysis;
- document organization;
- trend analysis;
- thematic summaries;
- search and navigation;
- qualitative research.

In [ ]:
import pandas as pd

topic_modeling_uses = pd.DataFrame(
    [
        ("Corpus exploration", "identify recurring themes"),
        ("Document organization", "group documents by topic mixture"),
        ("Trend analysis", "track topic prevalence over time"),
        ("Search support", "browse documents by inferred theme"),
        ("Qualitative analysis", "support human interpretation"),
    ],
    columns=["Use", "Purpose"],
)

topic_modeling_uses

A topic model does not discover objective meanings. It identifies statistical
patterns that researchers interpret as topics.

# 2. Topic Modeling Versus Classification

| Property | Topic Modeling | Classification |
|---|---|---|
| Supervision | usually unlabeled | labeled training data |
| Output | latent topic distributions | predefined class labels |
| Goal | discover structure | predict known categories |
| Interpretation | human naming required | class names already defined |
| Document membership | mixture of topics | often one predicted class |

Topic modeling also differs from hard clustering because one document can
contain several topics with different proportions.

# 3. The LDA Intuition

**Latent Dirichlet Allocation (LDA)** assumes:

- each document is a mixture of topics;
- each topic is a distribution over words.

Simplified example:

```text
Document A:
70% machine learning
20% data engineering
10% research methods
```

A topic might assign high probability to words such as:

```text
model, training, feature, classifier, accuracy
```

> **Key Idea**
>
> LDA returns probability distributions. A document is not restricted to one
> topic, and a word is not permanently assigned to one topic.

# 4. Example Corpus

The notebook uses a small synthetic corpus containing documents about:

- machine learning;
- finance;
- health;
- travel.

In [ ]:
documents = pd.DataFrame(
    [
        ("D01", "machine learning models require training data and evaluation"),
        ("D02", "neural networks learn features from large datasets"),
        ("D03", "classification accuracy depends on model validation"),
        ("D04", "feature engineering improves machine learning performance"),
        ("D05", "deep learning uses neural network layers and optimization"),
        ("D06", "the stock market moved after the earnings report"),
        ("D07", "investors evaluate risk return and portfolio performance"),
        ("D08", "bank interest rates affect loans and savings"),
        ("D09", "financial markets respond to inflation and policy"),
        ("D10", "the company reported revenue profit and expenses"),
        ("D11", "regular exercise supports heart health and fitness"),
        ("D12", "doctors recommend healthy food sleep and movement"),
        ("D13", "medical treatment depends on diagnosis and patient history"),
        ("D14", "nutrition and exercise reduce health risks"),
        ("D15", "the clinic provides patient care and medical advice"),
        ("D16", "travelers booked flights hotels and airport transport"),
        ("D17", "the tourism guide recommended museums and local restaurants"),
        ("D18", "passengers checked luggage before the international flight"),
        ("D19", "the hotel reservation included breakfast and city tours"),
        ("D20", "tourists planned a beach holiday and sightseeing trip"),
        ("D21", "machine learning helps analyze financial risk"),
        ("D22", "health researchers use statistical models and patient data"),
        ("D23", "travel companies predict demand using historical data"),
        ("D24", "banks use machine learning for fraud detection"),
    ],
    columns=["document_id", "text"],
)

documents.head()

In [ ]:
print("Number of documents:", len(documents))
print("Average words per document:", documents["text"].str.split().str.len().mean())

Mixed-theme documents are included so the model can represent topic mixtures.

# 5. Preparing the Document-Term Matrix

Scikit-learn LDA expects a nonnegative document-term matrix.

`CountVectorizer` creates token-count features.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(
    lowercase=True,
    stop_words="english",
    min_df=1,
    max_df=0.95,
)

document_term_matrix = vectorizer.fit_transform(
    documents["text"]
)

feature_names = vectorizer.get_feature_names_out()

print("Matrix shape:", document_term_matrix.shape)
print("Vocabulary size:", len(feature_names))
print("First 20 features:", feature_names[:20])

The matrix contains one row per document and one column per vocabulary term.

In [ ]:
document_term_frame = pd.DataFrame(
    document_term_matrix.toarray(),
    columns=feature_names,
    index=documents["document_id"],
)

document_term_frame.iloc[:5, :15]

# 6. Why LDA Uses Counts

LDA is a probabilistic model of token occurrences. Count features align more
directly with its assumptions than TF-IDF weights.

TF-IDF is valuable for retrieval and classification, but it is not the standard
input representation for classical LDA.

Preprocessing decisions still influence the count matrix:

- tokenization;
- stop words;
- stemming or lemmatization;
- n-grams;
- minimum and maximum document frequency;
- language-specific normalization.

# 7. Training an LDA Model

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation

lda_model = LatentDirichletAllocation(
    n_components=4,
    learning_method="batch",
    max_iter=30,
    random_state=42,
)

document_topic_matrix = lda_model.fit_transform(
    document_term_matrix
)

print("Document-topic shape:", document_topic_matrix.shape)
print("Topic-word shape:", lda_model.components_.shape)

`n_components=4` requests four latent topics. The model does not know the human
names of those topics.

# 8. Topic-Word Distributions

`lda_model.components_` contains learned topic-term weights.

Each row corresponds to one topic. Each column corresponds to one vocabulary
term.

In [ ]:
topic_word_weights = pd.DataFrame(
    lda_model.components_,
    columns=feature_names,
    index=[f"Topic {index}" for index in range(lda_model.n_components)],
)

topic_word_weights.iloc[:, :12]

The raw component values can be normalized to approximate word probabilities
within each topic.

In [ ]:
import numpy as np

topic_word_probabilities = (
    lda_model.components_
    / lda_model.components_.sum(axis=1, keepdims=True)
)

topic_probability_frame = pd.DataFrame(
    topic_word_probabilities,
    columns=feature_names,
    index=[f"Topic {index}" for index in range(lda_model.n_components)],
)

topic_probability_frame.iloc[:, :12].round(4)

# 9. Displaying Top Terms

Topics are commonly inspected through their highest-weight terms.

In [ ]:
def get_top_terms(
    model: LatentDirichletAllocation,
    terms,
    top_n: int = 10,
) -> pd.DataFrame:
    rows = []

    for topic_index, weights in enumerate(model.components_):
        top_indices = weights.argsort()[-top_n:][::-1]
        top_terms = [terms[index] for index in top_indices]
        top_weights = [weights[index] for index in top_indices]

        rows.append(
            {
                "topic": f"Topic {topic_index}",
                "top_terms": ", ".join(top_terms),
                "top_weights": [round(value, 3) for value in top_weights],
            }
        )

    return pd.DataFrame(rows)


top_terms_frame = get_top_terms(
    lda_model,
    feature_names,
    top_n=10,
)

top_terms_frame

Top terms support interpretation, but isolated term lists may be ambiguous.
Inspect representative documents as well.

# 10. Document-Topic Distributions

Each document receives a probability-like distribution over topics.

In [ ]:
document_topic_frame = pd.DataFrame(
    document_topic_matrix,
    columns=[
        f"Topic {index}"
        for index in range(lda_model.n_components)
    ],
)

document_topic_frame.insert(
    0,
    "document_id",
    documents["document_id"],
)

document_topic_frame.insert(
    1,
    "text",
    documents["text"],
)

document_topic_frame.head().round(3)

The topic values in each row sum to approximately 1.

In [ ]:
row_sums = document_topic_matrix.sum(axis=1)

print("Minimum row sum:", row_sums.min())
print("Maximum row sum:", row_sums.max())

# 11. Dominant Topic Assignment

A **dominant topic** is the topic with the highest value for a document.

In [ ]:
dominant_topic_indices = document_topic_matrix.argmax(axis=1)
dominant_topic_scores = document_topic_matrix.max(axis=1)

dominant_topic_frame = documents.copy()
dominant_topic_frame["dominant_topic"] = [
    f"Topic {index}"
    for index in dominant_topic_indices
]
dominant_topic_frame["dominant_score"] = dominant_topic_scores

dominant_topic_frame.head(10)

Reducing a mixture to one dominant topic is convenient but discards secondary
themes.

## 11.1 Representative Documents

In [ ]:
representative_rows = []

for topic_index in range(lda_model.n_components):
    best_document_index = document_topic_matrix[:, topic_index].argmax()

    representative_rows.append(
        {
            "topic": f"Topic {topic_index}",
            "document_id": documents.iloc[best_document_index]["document_id"],
            "score": document_topic_matrix[best_document_index, topic_index],
            "text": documents.iloc[best_document_index]["text"],
        }
    )

representative_documents = pd.DataFrame(representative_rows)
representative_documents

Representative documents help determine whether top terms form a coherent
theme.

# 12. Interpreting and Naming Topics

Topic names are assigned by humans after inspecting:

- top terms;
- high-scoring documents;
- corpus domain;
- topic overlap;
- model stability.

In [ ]:
proposed_topic_names = {
    "Topic 0": "Review manually",
    "Topic 1": "Review manually",
    "Topic 2": "Review manually",
    "Topic 3": "Review manually",
}

interpretation_table = top_terms_frame[
    ["topic", "top_terms"]
].copy()

interpretation_table["proposed_name"] = (
    interpretation_table["topic"]
    .map(proposed_topic_names)
)

interpretation_table

Avoid treating a short label as if it were generated by the model. The label
is a human interpretation.

# 13. Topic Prevalence

Average document-topic weight estimates how prevalent each topic is across the
corpus.

In [ ]:
topic_prevalence = pd.Series(
    document_topic_matrix.mean(axis=0),
    index=[
        f"Topic {index}"
        for index in range(lda_model.n_components)
    ],
    name="Average topic weight",
).sort_values(ascending=False)

topic_prevalence

Topic prevalence depends on the corpus, preprocessing, and model settings.

# 14. Choosing the Number of Topics

The number of topics is a hyperparameter.

Too few topics may merge distinct themes. Too many topics may create duplicate,
narrow, or unstable topics.

In [ ]:
model_comparison_rows = []

for topic_count in [2, 3, 4, 5, 6]:
    model = LatentDirichletAllocation(
        n_components=topic_count,
        learning_method="batch",
        max_iter=30,
        random_state=42,
    )

    model.fit(document_term_matrix)

    model_comparison_rows.append(
        {
            "n_topics": topic_count,
            "perplexity": model.perplexity(document_term_matrix),
            "log_likelihood": model.score(document_term_matrix),
        }
    )

topic_count_comparison = pd.DataFrame(model_comparison_rows)
topic_count_comparison.round(3)

Statistical scores should be combined with interpretability and stability.

# 15. Perplexity

Perplexity measures how well a probabilistic model predicts the observed data.

Lower perplexity is generally better under the same data and evaluation
procedure.

Perplexity has limitations:

- lower perplexity does not guarantee more interpretable topics;
- training perplexity may favor overly complex models;
- preprocessing strongly affects values;
- scores are not directly comparable across different corpora.

In [ ]:
training_perplexity = lda_model.perplexity(
    document_term_matrix
)

print(f"Training perplexity: {training_perplexity:.3f}")

A validation or held-out corpus provides a more meaningful evaluation than
training data alone.

# 16. Topic Coherence

Topic coherence attempts to measure whether high-ranking words in a topic tend
to occur together in a meaningful way.

Common coherence families include:

- UMass;
- UCI;
- NPMI;
- C_v.

Scikit-learn does not provide these metrics directly.

Coherence is not equivalent to human usefulness. A coherent topic may still be
irrelevant to the research question, while a useful rare topic may receive a
weaker score.

## 16.1 Simple Educational Co-Occurrence Score

The following function is a basic co-occurrence illustration, not a standard
coherence implementation.

In [ ]:
from itertools import combinations

binary_document_term = (
    document_term_matrix.toarray() > 0
).astype(int)

term_to_index = {
    term: index
    for index, term in enumerate(feature_names)
}


def simple_topic_cooccurrence(
    terms: list[str],
) -> float:
    pair_scores = []

    for left, right in combinations(terms, 2):
        left_index = term_to_index[left]
        right_index = term_to_index[right]

        joint_count = np.sum(
            (binary_document_term[:, left_index] == 1)
            & (binary_document_term[:, right_index] == 1)
        )

        pair_scores.append(joint_count)

    return (
        float(np.mean(pair_scores))
        if pair_scores
        else 0.0
    )


coherence_rows = []

for topic_index, weights in enumerate(lda_model.components_):
    top_indices = weights.argsort()[-5:][::-1]
    terms = [feature_names[index] for index in top_indices]

    coherence_rows.append(
        {
            "topic": f"Topic {topic_index}",
            "terms": ", ".join(terms),
            "simple_cooccurrence": simple_topic_cooccurrence(terms),
        }
    )

pd.DataFrame(coherence_rows)

Use established coherence implementations for formal experiments.

# 17. Topic Stability

Topic models may change when:

- the random seed changes;
- documents are added or removed;
- preprocessing changes;
- hyperparameters change;
- the vocabulary changes.

In [ ]:
stability_models = []

for seed in [1, 7, 42]:
    model = LatentDirichletAllocation(
        n_components=4,
        learning_method="batch",
        max_iter=30,
        random_state=seed,
    )

    model.fit(document_term_matrix)
    stability_models.append((seed, model))

In [ ]:
stability_summary = []

for seed, model in stability_models:
    terms = get_top_terms(
        model,
        feature_names,
        top_n=6,
    )

    for row in terms.itertuples(index=False):
        stability_summary.append(
            {
                "random_state": seed,
                "topic": row.topic,
                "top_terms": row.top_terms,
            }
        )

pd.DataFrame(stability_summary)

Topic numbers are arbitrary. `Topic 0` from one run does not necessarily match
`Topic 0` from another run. Topic alignment requires comparing term
distributions.

# 18. Hyperparameters

Important LDA hyperparameters include:

- `n_components`: number of topics;
- `doc_topic_prior`: document-topic concentration;
- `topic_word_prior`: topic-word concentration;
- `learning_method`: batch or online;
- `max_iter`: training iterations;
- `random_state`: reproducibility.

Lower concentration values often encourage sparser distributions, but the
effect should be interpreted and validated for the corpus.

In [ ]:
sparse_lda = LatentDirichletAllocation(
    n_components=4,
    doc_topic_prior=0.1,
    topic_word_prior=0.1,
    learning_method="batch",
    max_iter=30,
    random_state=42,
)

sparse_document_topics = sparse_lda.fit_transform(
    document_term_matrix
)

pd.DataFrame(
    sparse_document_topics[:5],
    columns=[f"Topic {i}" for i in range(4)],
).round(3)

# 19. Preprocessing Decisions

Topic quality is highly sensitive to preprocessing.

Important choices include:

- stop-word list;
- minimum document frequency;
- maximum document frequency;
- stemming or lemmatization;
- phrase detection;
- proper-name handling;
- number handling;
- domain terms;
- language-specific normalization.

In [ ]:
preprocessing_comparison = pd.DataFrame(
    [
        ("Remove common function words", "may improve interpretability"),
        ("Remove domain terms", "may erase meaningful topics"),
        ("Use stemming", "may merge inflected forms"),
        ("Use bigrams", "may preserve phrases"),
        ("Increase min_df", "removes rare words and rare topics"),
        ("Lowercase everything", "may merge named entities and common words"),
    ],
    columns=["Decision", "Possible effect"],
)

preprocessing_comparison

Preprocessing should be documented and tested through topic inspection.

# 20. New-Document Inference

A fitted LDA model can estimate topic mixtures for new documents.

The new text must be transformed with the fitted vectorizer.

In [ ]:
new_documents = [
    "the bank uses models to detect financial fraud",
    "the traveler booked a hotel near the beach",
    "exercise and nutrition improve patient health",
]

new_matrix = vectorizer.transform(new_documents)
new_topic_distributions = lda_model.transform(new_matrix)

new_topic_frame = pd.DataFrame(
    new_topic_distributions,
    columns=[
        f"Topic {index}"
        for index in range(lda_model.n_components)
    ],
    index=new_documents,
)

new_topic_frame.round(3)

Words absent from the fitted vocabulary cannot directly influence the topic
mixture.

# 21. Error Analysis and Common Failures

Common failures include:

- incoherent top terms;
- duplicate topics;
- topics defined by formatting or source;
- topics dominated by names;
- rare topics disappearing;
- broad topics splitting unnecessarily;
- mixed-language topics;
- unstable topic assignments;
- interpreting topic weights as certainty;
- naming topics too confidently.

In [ ]:
topic_model_errors = pd.DataFrame(
    [
        ("Topic contains unrelated frequent words", "stop-word or preprocessing issue"),
        ("Two topics have nearly identical terms", "too many topics or unstable solution"),
        ("One topic contains publisher names", "source metadata dominates content"),
        ("Rare theme is missing", "min_df too high or too few examples"),
        ("Mixed-language top terms", "language handling is insufficient"),
        ("Topic name is overly specific", "human interpretation overreaches"),
    ],
    columns=["Observed problem", "Possible cause"],
)

topic_model_errors

Topic modeling should be treated as exploratory analysis rather than an
automatic source of ground-truth categories.

# 22. Multilingual and Arabic Considerations

Multilingual topic modeling must consider:

- language identification;
- script;
- translation;
- multilingual stop words;
- code-switching;
- language-specific tokenization;
- shared versus separate vocabularies.

## 22.1 Arabic Topic Modeling

Arabic topic modeling may require decisions about:

- diacritics;
- Alef normalization;
- clitic segmentation;
- light stemming;
- root extraction;
- Modern Standard Arabic versus dialect;
- Arabizi;
- code-switching.

In [ ]:
arabic_documents = [
    "التعلم الآلي يعتمد على البيانات والنماذج",
    "تدريب النموذج يحتاج إلى تقييم دقيق",
    "الفاتورة تتضمن تفاصيل الدفع والاشتراك",
    "البنك يراجع القروض وأسعار الفائدة",
    "الصحة تتحسن مع الرياضة والغذاء",
    "الطبيب يراجع تاريخ المريض والعلاج",
    "السفر يشمل حجز الطيران والفندق",
    "السياحة تتضمن المتاحف والمطاعم المحلية",
]

arabic_vectorizer = CountVectorizer(
    token_pattern=r"(?u)\b\w+\b",
    min_df=1,
)

arabic_matrix = arabic_vectorizer.fit_transform(
    arabic_documents
)

arabic_lda = LatentDirichletAllocation(
    n_components=4,
    max_iter=30,
    random_state=42,
)

arabic_lda.fit(arabic_matrix)

arabic_top_terms = get_top_terms(
    arabic_lda,
    arabic_vectorizer.get_feature_names_out(),
    top_n=6,
)

arabic_top_terms

This small example demonstrates the mechanics only. Reliable Arabic topic
modeling requires larger, variety-aware corpora and careful linguistic review.

# 23. Reproducibility and Reporting

Report:

- corpus source and date;
- document count;
- preprocessing;
- vectorizer parameters;
- vocabulary size;
- number of topics;
- priors;
- learning method;
- iteration count;
- random seed;
- topic interpretation method;
- stability analysis;
- limitations.

In [ ]:
import sklearn

experiment_metadata = pd.Series(
    {
        "documents": len(documents),
        "vocabulary_size": len(feature_names),
        "n_topics": lda_model.n_components,
        "learning_method": lda_model.learning_method,
        "max_iter": lda_model.max_iter,
        "random_state": lda_model.random_state,
        "scikit-learn_version": sklearn.__version__,
    },
    name="LDA experiment",
)

experiment_metadata

# 24. Knowledge Check

1. What is topic modeling?
2. How does topic modeling differ from classification?
3. What are the two central distributions in LDA?
4. Why is a document represented as a topic mixture?
5. Why does classical LDA usually use count features?
6. What does `n_components` control?
7. What are topic-word distributions?
8. What are document-topic distributions?
9. Why should representative documents be inspected?
10. Why are topic names human interpretations?
11. What does perplexity measure?
12. Why may lower perplexity fail to produce better topics?
13. What is topic coherence?
14. Why should stability be tested?
15. Which Arabic preprocessing decisions affect topic modeling?

# 25. Exercises

## Exercise 1 — Corpus Preparation

Prepare a small corpus and justify stop-word, tokenization, and frequency
thresholds.

## Exercise 2 — Train LDA

Train models with 3, 5, and 7 topics.

## Exercise 3 — Topic Interpretation

List top terms and representative documents for every topic.

## Exercise 4 — Document Mixtures

Identify documents with strong topic mixtures rather than one dominant topic.

## Exercise 5 — Topic Prevalence

Calculate average topic prevalence and compare it across document groups.

## Exercise 6 — Model Comparison

Compare topic counts using perplexity, coherence, and human interpretation.

## Exercise 7 — Stability

Train the same model with five random seeds and compare top-term overlap.

## Exercise 8 — Arabic Topic Modeling

Compare Arabic topic models with and without light normalization.

## Challenge Exercises

1. Implement topic alignment across random seeds.
2. Add bigram phrase features before LDA.
3. Compare batch and online learning.
4. Track topic prevalence over time.
5. Build a report containing topic terms, representative documents, prevalence,
   and stability.

# 26. Summary and Next Lesson

In this lesson:

- topic modeling discovered latent statistical themes without class labels;
- LDA represented documents as topic mixtures;
- LDA represented topics as distributions over vocabulary terms;
- count-based document-term matrices matched the model's assumptions;
- top terms and representative documents supported interpretation;
- dominant-topic labels simplified mixtures but discarded secondary themes;
- topic prevalence summarized corpus-level patterns;
- topic count was treated as a hyperparameter;
- perplexity measured predictive fit but not necessarily interpretability;
- coherence and human review supported topic assessment;
- random seeds and preprocessing affected stability;
- new documents were transformed with the fitted vectorizer;
- Arabic topic modeling required language-specific normalization and review.

## Next Lesson

**Lesson 19: Sequence Labeling and Named Entity Recognition Foundations**
introduces token-level labels, BIO tagging, entity spans, evaluation, and
classical sequence-modeling concepts.

# References

- Blei, D. M., Ng, A. Y., & Jordan, M. I. *Latent Dirichlet Allocation*.
- Jurafsky, D., & Martin, J. H. *Speech and Language Processing*.
- scikit-learn LatentDirichletAllocation documentation.
- topic coherence and topic stability literature.
- Arabic topic-modeling literature.